# Selecting a high-quality sample of dwarf galaxies

In this notebook, we apply basic selection cuts to our dwarf galaxy catalogue. 

We start with our own catalogue of dwarf galaxies selected from the stellar Mass and Emission Line (MEL) DESI value added catalogue (VAC), including all dwarfs with CIGALE derived stellar masses less than 10^9 solar masses. 

Here, we further filter by some basic quality cuts:
- Including only main zurvey targets (`zcat_primary`)
- Removing duplicate sources (all duplicates appear to have the same flux measurements, arise from photometry?)
- `snr_median` > 2 (i.e. some measured continuum)
- SNR  > 3 for the BPT lines (\[O III\] 5007$\AA$, \[N II\] 6583$\AA$, H$\alpha$ and H$\beta$
- H$\alpha$ equivalent width greater than 14 (avoiding diffuse intergalactic medium contamination)

This yields a sample of XXX reasonably measured dwarf galaxies

In [ ]:
1

In [ ]:
from astropy.table import Table, join, unique
import numpy as np

In [ ]:
from pathlib import Path
import os

In [ ]:
outfile = Path("data/dwarfs_good.fits")

In [ ]:
if outfile.exists():
    os.remove(outfile)

## Load the catalogue

In [ ]:
dwarf_cat = Table.read("data/dwarfs_cigale.fits")

## Remove duplicated targets

In [ ]:
filt_primary = dwarf_cat["zcat_primary"] == b't'
print(len(dwarf_cat))
print(np.sum(filt_primary))
primary_cat = dwarf_cat[filt_primary]
primary_cat.sort("snr_med", reverse=True)

In [ ]:
primary_cat_unique = unique(primary_cat, keys="targetid")

In [ ]:
print(len(primary_cat_unique))

Very few values are duplicated

## Quality cuts

In [ ]:
required_lines = ["nii6583", "halpha", "oiii5007", "hbeta"]

filt = primary_cat_unique["zcat_primary"] == b't'

filt &= primary_cat_unique["fracflux_g"] < 0.3
filt &= primary_cat_unique["fracflux_r"] < 0.3
filt &= primary_cat_unique["fracflux_z"] < 0.3

print("fracflux", np.sum(filt))
for line in required_lines:
    filt &= primary_cat_unique[f"{line}_flux"] / primary_cat_unique[f"{line}_fluxerr"] > 3
    print(line, np.sum(filt))

filt &= primary_cat_unique["snr_med"] > 2
print("SNR", np.sum(filt))

filt &= primary_cat_unique["halpha_ew"] > 14
print("Halpha", np.sum(filt))

print(sum(filt))

In [ ]:
assert np.all(primary_cat_unique["zwarn"] == 0)
assert np.all(primary_cat_unique["spectype"] == b'GALAXY')

In [ ]:
good_dwarfs = primary_cat_unique[filt]

In [ ]:
np.max(good_dwarfs["fracflux_r"])

In [ ]:
good_dwarfs

## Write refined catalogue

In [ ]:
good_dwarfs.write(outfile)

# Analyzing duplicates

In [ ]:
u, c = np.unique(primary_cat["targetid"], return_counts=True)
duplicate_ids = u[c > 1]

In [ ]:
dwarf_cat.duplicated

In [ ]:
flux_cols = [col for col in primary_cat.columns if "_flux" in col]

In [ ]:
nonredundant_duplicate = False
for i in range(len(duplicated_ids)):
    all_duplicated = np.all(
        (primary_cat[primary_cat.targetid == duplicated_ids[i]])[[*flux_cols, "z"]].duplicated(keep=False)
    )
    
    nonredunant_duplicate = nonredunant_duplicate or not all_duplicated

    if not all_duplicated:
        print(duplicate_ids[i])
        


The above code checks whether any of the duplicated values (excluding other surveys) have different values of flux. Since there is no output, all of the duplicates are redundant f

# Selection plots

In [ ]:
import matplotlib.pyplot as plt

import arya
arya.init("mnras")


In [ ]:
len(good_dwarfs)

In [ ]:
bins = np.linspace(0, 40, 100)

plt.hist(dwarf_cat["snr_med"], bins=bins)
plt.hist(good_dwarfs["snr_med"], bins=bins, color="C1");
plt.xlabel("SNR median")
plt.ylabel("count")

In [ ]:
for line in required_lines:
    bins = np.linspace(0, 40, 100)
    
    plt.hist(dwarf_cat[f"{line}_flux"] / dwarf_cat[f"{line}_fluxerr"], bins=bins)
    plt.hist(good_dwarfs[f"{line}_flux"] / good_dwarfs[f"{line}_fluxerr"] , bins=bins, color="C1");
    plt.xlabel(f"SNR {line}")
    plt.ylabel("count")
    plt.yscale("log")
    plt.show()

In [ ]:
plt.scatter(dwarf_cat["halpha_flux"] / dwarf_cat["halpha_fluxerr"], dwarf_cat["snr_med"], s=0.3, alpha=0.01, lw=0, color="k")

plt.scatter(good_dwarfs["halpha_flux"] / good_dwarfs["halpha_fluxerr"], good_dwarfs["snr_med"], s=1, lw=0, alpha=0.05)

## BPT screening diagram

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log([NII] $\lambda$6583 / H$\alpha$)",
    ylabel = r"log([OIII] $\lambda$5007 / H$\beta$)",
    xlim = (-2, 0.5),
    ylim = (-1, 1.5),
)


x = np.log10(dwarf_cat["nii6583_flux"] / dwarf_cat["halpha_flux"])
y = np.log10(dwarf_cat["oiii5007_flux"] / dwarf_cat["hbeta_flux"])

plt.scatter(x, y, s=0.3, alpha=0.3, color="k", lw=0)


x = np.log10(good_dwarfs["nii6583_flux"] / good_dwarfs["halpha_flux"])
y = np.log10(good_dwarfs["oiii5007_flux"] / good_dwarfs["hbeta_flux"])

plt.scatter(x, y, s=0.3, alpha=0.3, color="C0", lw=0)


# x_model = np.linspace(-2, 0.15, 1000)
# y_model = 0.61 / (x_model - 0.15) + 1.4 # Eq. 1 from Zou+2024, +0.1 dex modelling uncertainty shift in both directions
# plt.plot(x_model, y_model)

x_model = np.linspace(-2, -0.05, 1000)
y_model = 0.61 / (x_model - 0.15 + 0.1) + 1.3 # the original Kauffmann+2003 separation
plt.plot(x_model, y_model)

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log($M_\star / M_\odot$)",
    ylabel = r"log(SFR / M$_\odot$ yr$^{-1}$)",
    xlim = (6, 9.6),
    ylim = (-3, 2),
)

x = dwarf_cat["logm_cigale"]
y = dwarf_cat["logsfr_cigale"] 

plt.scatter(x, y, s=0.1, color="black", alpha=0.03, lw=0)


x = good_dwarfs["logm_cigale"]
y = good_dwarfs["logsfr_cigale"] 

plt.scatter(x, y, s=0.1, color="C0", alpha=0.1, lw=0)

In [ ]:
eps_o_sun = 8.7

In [ ]:
Z_SUN = 0.014

In [ ]:
np.random.choice(good_dwarfs["targetid"])

In [ ]:
fig, ax = plt.subplots()
ax.set(
    xlabel = r"log($M_\star / M_\odot$)",
    ylabel = r"$\log(Z/Z_\odot)$ (CIGALE)",
    xlim = (6, 9.6),
    ylim = (-1.5, 0.5),
)

x = dwarf_cat["logm_cigale"]
y = np.log10(dwarf_cat["z_cg"] / Z_SUN)

plt.scatter(x, y, s=0.1, color="black", alpha=0.03, lw=0)


x = good_dwarfs["logm_cigale"]
y = np.log10(good_dwarfs["z_cg"] / Z_SUN)

plt.scatter(x, y, s=0.1, color="C1", alpha=0.1, lw=0)


plt.plot(x_model, y_model)